# 🏙️ CAIN-GAN: Automated Urban Site Planning

**Otomatik kentsel peyzaj tasarımı için GAN tabanlı PyTorch implementasyonu**

📚 Based on: *"Automated site planning using CAIN-GAN model"* (Automation in Construction, Elsevier 2024)

---

## 📋 İçindekiler
1. [Ortam Kurulumu](#setup)
2. [Repository Klonlama](#clone)
3. [Bağımlılıkları Yükleme](#deps)
4. [GPU Kontrolü](#gpu)
5. [Veri Hazırlama](#data)
6. [Model Test](#test)
7. [Eğitim Başlatma](#train)
8. [Görselleştirme](#viz)
9. [Inference](#inference)

---

## ⚙️ Çalıştırma Talimatları

1. **Runtime → Change runtime type → GPU (T4/A100)** seçin
2. Tüm hücreleri sırayla çalıştırın: **Runtime → Run all**
3. Eğitim için kendi verinizi `data/` klasörüne yükleyin

## 1️⃣ Ortam Kurulumu <a id='setup'></a>

In [ ]:
# Colab ortamı kontrol
import sys
import os

print(f"Python version: {sys.version}")
print(f"Current directory: {os.getcwd()}")

# Google Drive bağlantısı (opsiyonel - veri için)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\n✅ Google Drive mounted at /content/drive")
except ImportError:
    print("\n⚠️ Not running on Colab, skipping Drive mount")

## 2️⃣ Repository Klonlama <a id='clone'></a>

⚠️ **GitHub kullanıcı adınızı aşağıda güncelleyin!**

In [ ]:
# GitHub kullanıcı adı (kendi reponuzu kullanıyorsanız değiştirin)
GITHUB_USERNAME = "ilker-23"
REPO_NAME = "cain-gan-urban-design"

# Repository'i klonla
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd {REPO_NAME}

# Dosyaları listele
!ls -la

## 3️⃣ Bağımlılıkları Yükleme <a id='deps'></a>

In [ ]:
# Gerekli paketleri yükle
!pip install -q -r requirements.txt

print("\n✅ Tüm bağımlılıklar yüklendi!")

In [ ]:
# Paket versiyonlarını doğrula
import torch
import torchvision
import albumentations
import numpy as np
import PIL

print("📦 Yüklü Paket Versiyonları:")
print(f"  PyTorch:        {torch.__version__}")
print(f"  Torchvision:    {torchvision.__version__}")
print(f"  Albumentations: {albumentations.__version__}")
print(f"  NumPy:          {np.__version__}")
print(f"  Pillow:         {PIL.__version__}")

## 4️⃣ GPU Kontrolü <a id='gpu'></a>

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ CUDA mevcut!")
    print(f"  GPU sayısı:     {torch.cuda.device_count()}")
    print(f"  GPU adı:        {torch.cuda.get_device_name(0)}")
    print(f"  CUDA sürümü:    {torch.version.cuda}")
    print(f"  GPU belleği:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # nvidia-smi çıktısı
    print("\n📊 nvidia-smi:")
    !nvidia-smi | head -20
else:
    print("❌ CUDA bulunamadı! Runtime → Change runtime type → GPU seçin.")

## 5️⃣ Veri Hazırlama <a id='data'></a>

### Seçenek A: Kendi verilerinizi Google Drive'dan yükleyin

### Seçenek B: Test için synthetic (sahte) veri oluşturun

In [ ]:
# Veri dizin yapısını oluştur
import os
from pathlib import Path

DATA_ROOT = Path("./data")

subdirs = [
    "site_context",
    "planning_guidance",
    "neighboring_footprints",
    "mask",
    "footprint_target",
    "height_target",
]

for split in ["train", "val", "test"]:
    for subdir in subdirs:
        (DATA_ROOT / split / subdir).mkdir(parents=True, exist_ok=True)

print("📁 Dizin yapısı oluşturuldu:")
!tree data/ -L 3 2>/dev/null || find data/ -maxdepth 3 -type d | sort

In [ ]:
# 🧪 Test için sahte (synthetic) veri oluştur
# (Kendi verinizi kullanıyorsanız bu hücreyi atlayın)

import numpy as np
from PIL import Image

def generate_synthetic_sample(sample_id: str, split: str = "train"):
    """Sahte CAIN-GAN örneği oluştur (test amaçlı)."""
    
    # Site context (roads=1, vegetation=2, water=3)
    site_context = np.random.choice([0, 1, 2, 3], size=(256, 256), p=[0.6, 0.2, 0.15, 0.05])
    Image.fromarray(site_context.astype(np.uint8) * 85).save(
        f"data/{split}/site_context/{sample_id}.png"
    )
    
    # Planning guidance (one-hot, 4 channels but saved as RGB-like)
    planning = np.random.randint(0, 4, size=(256, 256))
    planning_rgb = np.zeros((256, 256, 3), dtype=np.uint8)
    planning_rgb[:, :, 0] = (planning == 0) * 255  # Residential
    planning_rgb[:, :, 1] = (planning == 1) * 255  # Commercial
    planning_rgb[:, :, 2] = (planning == 2) * 255  # Manufacturing
    Image.fromarray(planning_rgb).save(
        f"data/{split}/planning_guidance/{sample_id}.png"
    )
    
    # Neighboring footprints
    footprints = (np.random.rand(256, 256) > 0.7).astype(np.uint8) * 255
    Image.fromarray(footprints).save(
        f"data/{split}/neighboring_footprints/{sample_id}.png"
    )
    
    # Mask (center 64x64 region is design area)
    mask = np.ones((256, 256), dtype=np.uint8) * 255
    mask[96:160, 96:160] = 0
    Image.fromarray(mask).save(f"data/{split}/mask/{sample_id}.png")
    
    # Footprint target
    footprint_target = (np.random.rand(256, 256) > 0.6).astype(np.uint8) * 255
    Image.fromarray(footprint_target).save(
        f"data/{split}/footprint_target/{sample_id}.png"
    )
    
    # Height target
    height_target = np.random.randint(0, 255, size=(256, 256), dtype=np.uint8)
    Image.fromarray(height_target).save(
        f"data/{split}/height_target/{sample_id}.png"
    )


# Her split için örnekler oluştur
samples_per_split = {"train": 20, "val": 5, "test": 5}

for split, count in samples_per_split.items():
    for i in range(count):
        generate_synthetic_sample(f"sample_{i:04d}", split=split)
    print(f"✅ {count} örnek oluşturuldu: {split}/")

print("\n🎉 Sahte veri seti hazır!")

## 6️⃣ Model Test <a id='test'></a>

In [ ]:
# Dataset modülünü test et
from cain_dataset import CAINDataset

# Footprint stage dataset
dataset = CAINDataset(
    data_root="./data",
    split="train",
    image_size=256,
    stage="footprint",
    augment=True,
)

print(f"📊 Dataset boyutu: {len(dataset)}")

# Bir örnek al
sample = dataset[0]
print(f"\n📦 Örnek anahtarları: {list(sample.keys())}")
print(f"  Conditional inputs: {sample['conditional_inputs'].shape}")
print(f"  Footprint:          {sample['footprint'].shape}")
print(f"  Sample ID:          {sample['sample_id']}")

In [ ]:
# Model mimarisini test et
import torch
from cain_architecture import CAINGANModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {device}")

# Modeli oluştur
conditional_channels = sample['conditional_inputs'].shape[0]
model = CAINGANModel(conditional_channels=conditional_channels).to(device)

# Parametre sayısı
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Toplam parametre: {total_params:,}")

# Forward pass testi
x = torch.randn(2, conditional_channels, 256, 256).to(device)

with torch.no_grad():
    footprint_out = model.forward_footprint(x)
    height_out = model.forward_height(x)

print(f"\n✅ Footprint generator: {x.shape} → {footprint_out.shape}")
print(f"✅ Height generator:    {x.shape} → {height_out.shape}")

## 7️⃣ Eğitim Başlatma <a id='train'></a>

⏰ **Süre tahmini:**
- Synthetic data (50 örnek): ~10-20 dakika
- Gerçek veri (1000+ örnek): ~2-6 saat (GPU'ya göre değişir)

In [ ]:
# Eğitim parametreleri (Colab GPU için optimize edilmiş)
TRAINING_CONFIG = {
    "data_root": "./data",
    "checkpoint_dir": "./checkpoints",
    "batch_size": 8,             # T4 GPU için (A100'de 16-32 yapabilirsiniz)
    "num_workers": 2,            # Colab CPU kısıtlaması
    "learning_rate": 0.0002,
    "epochs_footprint": 10,      # Test için az tutuldu (gerçek için 50+)
    "epochs_height": 10,         # Test için az tutuldu (gerçek için 50+)
    "save_interval": 5,
}

print("⚙️ Eğitim Konfigürasyonu:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Eğitimi başlat
from cain_training import CAINTrainer

trainer = CAINTrainer(
    data_root=TRAINING_CONFIG["data_root"],
    checkpoint_dir=TRAINING_CONFIG["checkpoint_dir"],
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=TRAINING_CONFIG["batch_size"],
    num_workers=TRAINING_CONFIG["num_workers"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
)

# İki aşamalı eğitim
trainer.train(
    num_epochs_footprint=TRAINING_CONFIG["epochs_footprint"],
    num_epochs_height=TRAINING_CONFIG["epochs_height"],
    save_interval=TRAINING_CONFIG["save_interval"],
)

## 8️⃣ Görselleştirme <a id='viz'></a>

In [ ]:
import matplotlib.pyplot as plt
import torch
from cain_dataset import CAINDataset

# Validation dataset
val_dataset = CAINDataset(
    data_root="./data",
    split="val",
    stage="footprint",
    augment=False,
)

# Bir örnek al
sample = val_dataset[0]
conditional = sample["conditional_inputs"].unsqueeze(0).to(device)
target = sample["footprint"]

# Model ile tahmin
trainer.generator_f.eval()
with torch.no_grad():
    prediction = trainer.generator_f(conditional)

# Görselleştir
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Conditional input (ilk kanal)
axes[0].imshow(conditional[0, 0].cpu().numpy(), cmap='gray')
axes[0].set_title('Input (Site Context)')
axes[0].axis('off')

# Ground truth
axes[1].imshow(target[0].cpu().numpy(), cmap='gray')
axes[1].set_title('Ground Truth Footprint')
axes[1].axis('off')

# Prediction
axes[2].imshow(prediction[0, 0].cpu().numpy(), cmap='gray')
axes[2].set_title('Generated Footprint')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('results_footprint.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Sonuç kaydedildi: results_footprint.png")

## 9️⃣ Inference (Yeni Tahminler) <a id='inference'></a>

In [ ]:
# Yeni bir site için tahmin yap
import torch
from PIL import Image
import numpy as np

def predict_new_site(
    site_context_path: str,
    planning_guidance_path: str,
    neighboring_footprints_path: str,
    mask_path: str,
    model_trainer,
):
    """Yeni bir site için footprint ve height tahmin et."""
    
    # Veriyi yükle ve preprocess et
    def load_and_preprocess(path, expand_channel=True):
        img = Image.open(path)
        if img.mode != "RGB" and img.mode != "L":
            img = img.convert("L")
        arr = np.array(img.resize((256, 256))) / 255.0
        if arr.ndim == 2 and expand_channel:
            arr = np.expand_dims(arr, axis=0)
        elif arr.ndim == 3:
            arr = arr.transpose(2, 0, 1)
        return arr.astype(np.float32)
    
    site = load_and_preprocess(site_context_path)
    planning = load_and_preprocess(planning_guidance_path)
    footprints = load_and_preprocess(neighboring_footprints_path)
    mask = load_and_preprocess(mask_path)
    
    # Birleştir
    conditional = np.concatenate([site, planning, footprints, mask], axis=0)
    conditional = torch.from_numpy(conditional).unsqueeze(0).to(device)
    
    # Stage 1: Footprint
    model_trainer.generator_f.eval()
    with torch.no_grad():
        footprint = model_trainer.generator_f(conditional)
    
    # Stage 2: Height
    model_trainer.generator_h.eval()
    with torch.no_grad():
        height = model_trainer.generator_h(conditional)
    
    return footprint.cpu().numpy()[0, 0], height.cpu().numpy()[0, 0]


# Test örneği
footprint_pred, height_pred = predict_new_site(
    "./data/test/site_context/sample_0000.png",
    "./data/test/planning_guidance/sample_0000.png",
    "./data/test/neighboring_footprints/sample_0000.png",
    "./data/test/mask/sample_0000.png",
    trainer,
)

# Görselleştir
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(footprint_pred, cmap='gray')
axes[0].set_title('Predicted Footprint')
axes[0].axis('off')

axes[1].imshow(height_pred, cmap='viridis')
axes[1].set_title('Predicted Heights')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Tahmin tamamlandı!")

## 💾 Checkpoint'leri Google Drive'a Kaydet

In [ ]:
# Sonuçları Drive'a kopyala
import shutil
from pathlib import Path

DRIVE_OUTPUT = "/content/drive/MyDrive/cain_gan_outputs"

if Path("/content/drive/MyDrive").exists():
    Path(DRIVE_OUTPUT).mkdir(exist_ok=True)
    
    # Checkpoint'leri kopyala
    if Path("./checkpoints").exists():
        shutil.copytree("./checkpoints", f"{DRIVE_OUTPUT}/checkpoints", dirs_exist_ok=True)
        print(f"✅ Checkpoint'ler kopyalandı: {DRIVE_OUTPUT}/checkpoints")
    
    # Sonuç görsellerini kopyala
    for img_file in Path(".").glob("results_*.png"):
        shutil.copy(img_file, f"{DRIVE_OUTPUT}/{img_file.name}")
        print(f"✅ Kopyalandı: {img_file.name}")
else:
    print("⚠️ Google Drive bağlı değil. İlk hücreyi çalıştırın.")

## 🎯 Sonraki Adımlar

### Üretim Kalitesinde Eğitim İçin:

1. **Gerçek veri toplayın:**
   - Uydu görüntüleri (Google Earth Engine, OpenStreetMap)
   - Bina footprint'leri (OSM, şehir açık verisi)
   - Planlama verisi (zoning, land use)

2. **Veriyi preprocess edin:**
   - 256×256 boyutuna getirin
   - Multi-channel encoding uygulayın
   - Train/val/test split (70/15/15)

3. **Tam eğitim çalıştırın:**
   ```python
   trainer.train(
       num_epochs_footprint=50,  # veya 100
       num_epochs_height=50,     # veya 100
   )
   ```

4. **Hiperparametre optimizasyonu:**
   - Learning rate scheduling
   - Loss weight tuning (λ_rec, λ_adv)
   - Batch size ayarı

5. **Değerlendirme metrikleri:**
   - FID (Fréchet Inception Distance)
   - LPIPS (Perceptual similarity)
   - SSIM, PSNR
   - Domain-specific metrics (urban coherence)

---

## 📚 Dökümantasyon

- `CAIN_GAN_IMPLEMENTATION.md` - Detaylı teknik dökümantasyon
- `QUICKSTART.md` - Hızlı başlangıç rehberi
- `README.md` - Proje genel bilgileri

## 📖 Referans

```
Jiang, F., Ma, J., Webster, C. J., Wang, W., & Cheng, J. C. (2024). 
Automated site planning using CAIN-GAN model. 
Automation in Construction, 159, 105286.
```

---

**🎓 Başarılar!** Tez ve makale çalışmanızda kolaylıklar dilerim. 🚀